
# Image Enhancement Based on Filtering — Google Colab Implementation

**Course topic:** *Image Enhancement Based on Filtering* (Spatial-domain and
Frequency-domain filtering)

This notebook implements and demonstrates image-enhancement techniques for
several common real-world image problems:

| Problem | Technique(s) applied |
|---|---|
| Dark / underexposed image | Gamma correction, Histogram Equalization |
| Bright / overexposed image | Gamma correction, Contrast stretching |
| Low-contrast image | Contrast stretching, Histogram Equalization, CLAHE |
| Blurred image | Unsharp masking, Laplacian sharpening, Frequency-domain high-pass boosting |
| Noisy image (salt & pepper) | Median filter, (compared against Mean & Gaussian filters) |

It also implements the **frequency-domain (Fourier / DFT-based)** filters
covered in the lecture slides: Ideal, Gaussian, and Butterworth low-pass and
high-pass filters, plus an `auto_enhance()` function that automatically
diagnoses an arbitrary input image and applies the most suitable correction.

> **How to run:** `Runtime → Run all`. No external downloads are required —
> a synthetic test image and its degraded variants are generated
> programmatically, so results are 100% reproducible. At the bottom of the
> notebook you can also upload **your own photo** and run `auto_enhance()`
> on it.


## 0. Setup

In [ ]:
!pip -q install scipy scikit-image opencv-python-headless matplotlib pillow numpy

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

np.random.seed(42)
os.makedirs("images/input", exist_ok=True)
os.makedirs("images/output", exist_ok=True)
print("Setup complete.")


## 1. Core enhancement library

The cell below defines every enhancement function used in this notebook —
identical to `src/enhancement.py` in the project repository — covering:

- **Point processing:** gamma correction, contrast stretching, histogram
  equalization, CLAHE
- **Spatial smoothing:** mean filter, Gaussian filter, median filter
- **Spatial sharpening:** Laplacian filter, unsharp masking
- **Frequency domain:** 2D FFT, Ideal / Gaussian / Butterworth low-pass and
  high-pass filters, spectrum visualization
- **Automatic diagnosis + routing:** `diagnose()` and `auto_enhance()`


In [ ]:
"""
enhancement.py
================
Image Enhancement Based on Filtering
-------------------------------------
Implements the spatial-domain and frequency-domain techniques covered in
"Image Enhancement Based on Filtering" (Wahyono, Ph.D., UGM):

Spatial domain:
    - Linear convolution filtering (generic kernel convolution)
    - Smoothing filters: mean (box) filter, Gaussian filter
    - Median filter (non-linear, local-statistic filter)
    - Sharpening: Laplacian filter, unsharp masking (F + alpha*(F - blur(F)))
    - Point-processing: gamma correction, contrast stretching,
      histogram equalization (for dark / bright / low-contrast images)

Frequency domain:
    - 2D DFT / FFT
    - Ideal Low-Pass / High-Pass Filter (ILPF / IHPF)
    - Gaussian Low-Pass / High-Pass Filter (GLPF / GHPF)
    - Butterworth Low-Pass / High-Pass Filter (BLPF / BHPF)
    - Laplacian-in-frequency-domain sharpening

Also includes:
    - Automatic problem detection (dark / bright / low-contrast / blurred /
      noisy) so a single `auto_enhance()` call can route an arbitrary image
      to the most suitable technique.
    - Simple quality metrics (PSNR, contrast/entropy) for before/after
      comparison.

Author: (project generated for course assignment on Image Enhancement)
"""

import numpy as np
from scipy import ndimage
from scipy.signal import convolve2d


# --------------------------------------------------------------------------
# 0. Utility functions
# --------------------------------------------------------------------------

def to_float(img):
    """Convert an image (uint8, 0-255) to float64 in range [0, 255]."""
    return img.astype(np.float64)


def to_uint8(img):
    """Clip to [0,255] and convert back to uint8."""
    return np.clip(img, 0, 255).astype(np.uint8)


def to_gray(img):
    """Convert an RGB image to grayscale (if needed)."""
    img = np.asarray(img)
    if img.ndim == 3:
        return (0.299 * img[..., 0] + 0.587 * img[..., 1] +
                0.114 * img[..., 2])
    return img.astype(np.float64)


def psnr(original, processed):
    """Peak Signal-to-Noise Ratio between two same-shape images."""
    original = to_float(original)
    processed = to_float(processed)
    mse = np.mean((original - processed) ** 2)
    if mse == 0:
        return float("inf")
    return 20 * np.log10(255.0 / np.sqrt(mse))


def entropy(img):
    """Shannon entropy of the image histogram - a proxy for information/contrast."""
    gray = to_gray(img).astype(np.uint8)
    hist, _ = np.histogram(gray, bins=256, range=(0, 255))
    prob = hist / hist.sum()
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))


def rms_contrast(img):
    """RMS contrast = standard deviation of pixel intensities."""
    return float(np.std(to_gray(img)))


# --------------------------------------------------------------------------
# 1. Generic spatial convolution (as taught: rotate kernel 180deg then slide)
# --------------------------------------------------------------------------

def apply_kernel(img, kernel, per_channel=True):
    """
    Direct implementation of the convolution shown in the slides:
        G = H * F  (kernel rotated 180 deg, slid across image)
    Uses scipy.signal.convolve2d with 'same' boundary handling for speed,
    matching the mathematical definition G[i,j] = sum H[u,v] F[i-u, j-v].
    """
    kernel = np.asarray(kernel, dtype=np.float64)
    img = to_float(img)

    if img.ndim == 2 or not per_channel:
        return convolve2d(img, kernel, mode="same", boundary="symm")

    out = np.zeros_like(img)
    for c in range(img.shape[2]):
        out[..., c] = convolve2d(img[..., c], kernel, mode="same", boundary="symm")
    return out


# --------------------------------------------------------------------------
# 2. Point processing: for DARK / BRIGHT / LOW-CONTRAST images
# --------------------------------------------------------------------------

def gamma_correction(img, gamma):
    """
    Power-law (gamma) transform: g = 255 * (f/255)^gamma
        gamma < 1  -> brightens dark images (expands dark tones)
        gamma > 1  -> darkens bright / washed-out images
    """
    img = to_float(img)
    normalized = img / 255.0
    corrected = np.power(normalized, gamma) * 255.0
    return to_uint8(corrected)


def contrast_stretching(img, low_pct=2, high_pct=98):
    """
    Linear contrast stretching (min-max / percentile stretch).
    Maps the [low_pct, high_pct] percentile range of intensities to [0,255].
    Ideal for LOW-CONTRAST images whose histogram occupies a narrow band.
    """
    img = to_float(img)
    if img.ndim == 3:
        out = np.zeros_like(img)
        for c in range(img.shape[2]):
            lo, hi = np.percentile(img[..., c], [low_pct, high_pct])
            if hi - lo < 1e-6:
                hi = lo + 1
            out[..., c] = (img[..., c] - lo) * 255.0 / (hi - lo)
        return to_uint8(out)
    else:
        lo, hi = np.percentile(img, [low_pct, high_pct])
        if hi - lo < 1e-6:
            hi = lo + 1
        out = (img - lo) * 255.0 / (hi - lo)
        return to_uint8(out)


def histogram_equalization(img):
    """
    Classic global histogram equalization.
    For color images, equalizes the luminance (Y) channel only, in the
    YCrCb color space, to avoid color-shift artifacts.
    """
    img = np.asarray(img)
    if img.ndim == 2:
        return _equalize_channel(img)

    # Color: work in YCrCb, equalize Y, convert back
    import cv2
    ycrcb = cv2.cvtColor(to_uint8(img), cv2.COLOR_RGB2YCrCb)
    ycrcb[..., 0] = _equalize_channel(ycrcb[..., 0])
    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)


def _equalize_channel(channel):
    channel = to_uint8(channel)
    hist, bins = np.histogram(channel.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    cdf_masked = np.ma.masked_equal(cdf, 0)
    cdf_masked = (cdf_masked - cdf_masked.min()) * 255 / \
                 (cdf_masked.max() - cdf_masked.min())
    cdf_final = np.ma.filled(cdf_masked, 0).astype("uint8")
    return cdf_final[channel]


def clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """
    Contrast Limited Adaptive Histogram Equalization.
    Better than global HE for images with uneven / local lighting
    (e.g. partially shadowed low-contrast images).
    """
    import cv2
    img = np.asarray(img)
    clahe_op = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    if img.ndim == 2:
        return clahe_op.apply(to_uint8(img))
    ycrcb = cv2.cvtColor(to_uint8(img), cv2.COLOR_RGB2YCrCb)
    ycrcb[..., 0] = clahe_op.apply(ycrcb[..., 0])
    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)


# --------------------------------------------------------------------------
# 3. Spatial smoothing (noise removal) & sharpening (deblurring)
# --------------------------------------------------------------------------

def mean_filter(img, ksize=3):
    """Averaging/box filter - simple smoothing (slide: 'Smoothing Linear Filters')."""
    kernel = np.ones((ksize, ksize)) / (ksize * ksize)
    return to_uint8(apply_kernel(img, kernel))


def gaussian_filter(img, sigma=1.0):
    """
    Gaussian smoothing filter: G_sigma = 1/(2*pi*sigma^2) * exp(-(x^2+y^2)/2*sigma^2)
    Removes high-frequency noise (low-pass) while preserving structure
    better than the mean filter.
    """
    img = to_float(img)
    if img.ndim == 3:
        out = np.zeros_like(img)
        for c in range(img.shape[2]):
            out[..., c] = ndimage.gaussian_filter(img[..., c], sigma=sigma)
        return to_uint8(out)
    return to_uint8(ndimage.gaussian_filter(img, sigma=sigma))


def median_filter(img, ksize=3):
    """
    Median filter - a local-statistic filter, excellent for removing
    impulsive (salt-and-pepper) noise without excessive blurring.
    """
    img = np.asarray(img)
    if img.ndim == 3:
        out = np.zeros_like(img)
        for c in range(img.shape[2]):
            out[..., c] = ndimage.median_filter(img[..., c], size=ksize)
        return out
    return ndimage.median_filter(img, size=ksize)


def laplacian_sharpen(img, weight=1.0):
    """
    Laplacian sharpening, following the slide's 2D Laplacian kernel
    [[0,1,0],[1,-4,1],[0,1,0]]:
        g(x,y) = f(x,y) - w * Laplacian(f)   (center coefficient negative)
    """
    kernel = np.array([[0, 1, 0],
                        [1, -4, 1],
                        [0, 1, 0]], dtype=np.float64)
    lap = apply_kernel(img, kernel)
    sharpened = to_float(img) - weight * lap
    return to_uint8(sharpened)


def unsharp_mask(img, sigma=2.0, amount=1.0):
    """
    Unsharp masking, exactly the 'Sharpening revisited' slide:
        detail = original - blurred
        sharpened = original + alpha * detail
    Effective for BLURRED images: recovers edges/detail lost to blur.
    """
    img = to_float(img)
    blurred = to_float(gaussian_filter(to_uint8(img), sigma=sigma))
    detail = img - blurred
    sharpened = img + amount * detail
    return to_uint8(sharpened)


# --------------------------------------------------------------------------
# 4. Frequency-domain filtering (2D FFT based)
# --------------------------------------------------------------------------

def _distance_grid(shape):
    """D(u,v): distance of every frequency-domain point from the DC center."""
    rows, cols = shape
    u = np.arange(rows) - rows // 2
    v = np.arange(cols) - cols // 2
    V, U = np.meshgrid(v, u)
    return np.sqrt(U ** 2 + V ** 2)


def ideal_filter(shape, d0, highpass=False):
    D = _distance_grid(shape)
    H = (D > d0).astype(np.float64) if highpass else (D <= d0).astype(np.float64)
    return H


def gaussian_freq_filter(shape, d0, highpass=False):
    D = _distance_grid(shape)
    H = np.exp(-(D ** 2) / (2 * (d0 ** 2)))
    return 1 - H if highpass else H


def butterworth_freq_filter(shape, d0, order=2, highpass=False):
    D = _distance_grid(shape)
    D[D == 0] = 1e-6  # avoid divide-by-zero at DC for highpass
    if highpass:
        H = 1 / (1 + (d0 / D) ** (2 * order))
    else:
        H = 1 / (1 + (D / d0) ** (2 * order))
    return H


def apply_freq_filter(img_gray, H):
    """
    Basic steps for filtering in the frequency domain (as in slide
    'Filtering in Frequency Domain: Basic Steps'):
        1-2. F(u,v) = FFT{f(x,y)}, shifted so DC is centered
        3.   G(u,v) = F(u,v) * H(u,v)
        4-5. g(x,y) = Re{ IFFT{ G(u,v) } }
    """
    F = np.fft.fftshift(np.fft.fft2(img_gray))
    G = F * H
    g = np.fft.ifft2(np.fft.ifftshift(G))
    return np.real(g)


def freq_lowpass(img, d0=40, kind="gaussian", order=2):
    """Apply a frequency-domain low-pass filter (denoise / smooth)."""
    gray = to_gray(img)
    if kind == "ideal":
        H = ideal_filter(gray.shape, d0)
    elif kind == "butterworth":
        H = butterworth_freq_filter(gray.shape, d0, order=order)
    else:
        H = gaussian_freq_filter(gray.shape, d0)
    return to_uint8(apply_freq_filter(gray, H))


def freq_highpass(img, d0=30, kind="gaussian", order=2, boost=1.0):
    """
    Apply a frequency-domain high-pass filter and add it back to the
    original (high-frequency emphasis) to sharpen a blurred image.
    """
    gray = to_gray(img)
    if kind == "ideal":
        H = ideal_filter(gray.shape, d0, highpass=True)
    elif kind == "butterworth":
        H = butterworth_freq_filter(gray.shape, d0, order=order, highpass=True)
    else:
        H = gaussian_freq_filter(gray.shape, d0, highpass=True)
    high = apply_freq_filter(gray, H)
    sharpened = gray + boost * high
    return to_uint8(sharpened)


def freq_spectrum_image(img):
    """Return a log-scaled magnitude spectrum, useful for visualization."""
    gray = to_gray(img)
    F = np.fft.fftshift(np.fft.fft2(gray))
    mag = np.log1p(np.abs(F))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-9) * 255
    return mag.astype(np.uint8)


# --------------------------------------------------------------------------
# 5. Automatic diagnosis + routing
# --------------------------------------------------------------------------

def diagnose(img):
    """
    Very simple heuristic diagnosis of the dominant defect in an image,
    based on mean brightness, RMS contrast, and high-frequency energy
    (Laplacian variance, a common blur metric).
    Returns one of: 'dark', 'bright', 'low_contrast', 'blurred', 'noisy', 'ok'
    """
    gray = to_gray(img)
    mean_brightness = gray.mean()
    contrast = gray.std()
    lap_var = ndimage.laplace(gray).var()  # low -> blurry
    # crude noise estimate: high-frequency energy from a median-filtered residual
    residual = gray - ndimage.median_filter(gray, size=3)
    noise_estimate = residual.std()

    if mean_brightness < 80:
        return "dark"
    if mean_brightness > 180:
        return "bright"
    if contrast < 35:
        return "low_contrast"
    if noise_estimate > 18:
        return "noisy"
    if lap_var < 120:
        return "blurred"
    return "ok"


def auto_enhance(img, verbose=True):
    """
    Detects the dominant problem in `img` and applies the most appropriate
    enhancement pipeline. Returns (enhanced_image, diagnosis_string).
    """
    problem = diagnose(img)
    if problem == "dark":
        out = gamma_correction(img, gamma=0.5)
    elif problem == "bright":
        out = gamma_correction(img, gamma=1.8)
    elif problem == "low_contrast":
        out = clahe(img, clip_limit=2.5)
    elif problem == "noisy":
        out = median_filter(img, ksize=3)
    elif problem == "blurred":
        out = unsharp_mask(img, sigma=2.0, amount=1.5)
    else:
        out = img
    if verbose:
        print(f"[auto_enhance] Detected problem: '{problem}' -> "
              f"applied {'no change' if problem == 'ok' else problem + ' correction'}")
    return out, problem


## 2. Generate synthetic test images

To make the notebook fully self-contained and reproducible (no external
image downloads needed), we synthesize one clean "base" image and derive
five degraded versions of it: **dark, bright, low-contrast, blurred,** and
**noisy (salt & pepper)**.

> Want to test your *own* image instead? Skip to **Section 8** near the end
> of the notebook.


In [ ]:
"""
make_test_images.py
====================
Generates a synthetic "clean" base image (so the project is fully
self-contained and reproducible without needing external downloads),
then derives 5 degraded test images that represent the classic problem
categories requested:

    1. dark.png          - underexposed image
    2. bright.png        - overexposed / washed-out image
    3. low_contrast.png  - compressed dynamic range
    4. blurred.png       - Gaussian-blurred (defocus-like) image
    5. noisy.png         - salt-and-pepper noise (impulsive noise)

Run this once to populate images/input/. Everything downstream
(the notebook, the report figures) is generated from these files, so
re-running is fully reproducible.
"""

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os

np.random.seed(42)

OUT_DIR = "images/input"
os.makedirs(OUT_DIR, exist_ok=True)


def make_base_image(size=(512, 512)):
    """
    Build a synthetic scene with a mix of smooth gradients, hard edges,
    circles, and text -- good for testing both smoothing and sharpening,
    since it contains flat regions, edges, and fine detail.
    """
    w, h = size
    img = Image.new("RGB", size, (30, 30, 40))
    draw = ImageDraw.Draw(img)

    # Sky-like vertical gradient (smooth region -> tests noise/smoothing)
    for y in range(h // 2):
        t = y / (h / 2)
        color = (int(40 + 120 * t), int(80 + 100 * t), int(160 + 60 * t))
        draw.line([(0, y), (w, y)], fill=color)

    # Ground gradient
    for y in range(h // 2, h):
        t = (y - h / 2) / (h / 2)
        color = (int(60 - 20 * t), int(110 - 40 * t), int(60 - 20 * t))
        draw.line([(0, y), (w, y)], fill=color)

    # "Sun" - circle with sharp edge (edge/contrast test)
    draw.ellipse([w - 160, 40, w - 60, 140], fill=(255, 230, 150))

    # Building silhouettes (hard vertical/horizontal edges -> sharpening test)
    rng = np.random.RandomState(1)
    x = 30
    while x < w - 40:
        bw = rng.randint(30, 70)
        bh = rng.randint(80, 220)
        draw.rectangle([x, h // 2 - bh, x + bw, h // 2], fill=(20, 20, 25))
        # windows (fine detail -> tests blur/sharpen quality)
        for wy in range(h // 2 - bh + 10, h // 2 - 10, 18):
            for wx in range(x + 6, x + bw - 6, 14):
                if rng.rand() > 0.4:
                    draw.rectangle([wx, wy, wx + 6, wy + 10], fill=(240, 220, 120))
        x += bw + rng.randint(10, 25)

    # High-frequency texture patch (checkerboard) -> stresses low-pass/high-pass filters
    tex_x0, tex_y0, tex_size, cell = 20, h - 140, 120, 8
    for i in range(tex_size // cell):
        for j in range(tex_size // cell):
            if (i + j) % 2 == 0:
                draw.rectangle([tex_x0 + i * cell, tex_y0 + j * cell,
                                 tex_x0 + (i + 1) * cell, tex_y0 + (j + 1) * cell],
                                fill=(230, 230, 230))
            else:
                draw.rectangle([tex_x0 + i * cell, tex_y0 + j * cell,
                                 tex_x0 + (i + 1) * cell, tex_y0 + (j + 1) * cell],
                                fill=(10, 10, 10))

    # Text (fine detail, good for visually judging sharpening/blur)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    draw.text((w // 2 - 90, h - 40), "IMAGE ENHANCEMENT", fill=(255, 255, 255), font=font)

    return np.array(img)


def make_dark(img, factor=0.28):
    """Simulate underexposure: multiply intensities down."""
    out = (img.astype(np.float64) * factor).clip(0, 255).astype(np.uint8)
    return out


def make_bright(img, factor=1.9, offset=60):
    """Simulate overexposure: scale up and add offset, causing clipping (washed out)."""
    out = (img.astype(np.float64) * factor + offset).clip(0, 255).astype(np.uint8)
    return out


def make_low_contrast(img, low=90, high=150):
    """Compress the dynamic range into a narrow band [low, high]."""
    img = img.astype(np.float64)
    normalized = (img - img.min()) / (img.max() - img.min())
    out = (normalized * (high - low) + low).clip(0, 255).astype(np.uint8)
    return out


def make_blurred(img, sigma=3.5):
    from scipy import ndimage
    out = np.zeros_like(img)
    for c in range(img.shape[2]):
        out[..., c] = ndimage.gaussian_filter(img[..., c], sigma=sigma)
    return out


def make_noisy(img, amount=0.06):
    """Add salt-and-pepper impulsive noise."""
    out = img.copy()
    h, w = img.shape[:2]
    n_salt = int(amount * h * w / 2)
    n_pepper = int(amount * h * w / 2)

    rng = np.random.RandomState(7)
    ys = rng.randint(0, h, n_salt)
    xs = rng.randint(0, w, n_salt)
    out[ys, xs] = 255

    ys = rng.randint(0, h, n_pepper)
    xs = rng.randint(0, w, n_pepper)
    out[ys, xs] = 0
    return out


def main():
    base = make_base_image()
    Image.fromarray(base).save(os.path.join(OUT_DIR, "original.png"))

    variants = {
        "dark.png": make_dark(base),
        "bright.png": make_bright(base),
        "low_contrast.png": make_low_contrast(base),
        "blurred.png": make_blurred(base),
        "noisy.png": make_noisy(base),
    }
    for name, arr in variants.items():
        Image.fromarray(arr).save(os.path.join(OUT_DIR, name))
        print("saved", name)





In [ ]:
base = make_base_image()
Image.fromarray(base).save("images/input/original.png")

variants = {
    "dark.png": make_dark(base),
    "bright.png": make_bright(base),
    "low_contrast.png": make_low_contrast(base),
    "blurred.png": make_blurred(base),
    "noisy.png": make_noisy(base),
}
for name, arr in variants.items():
    Image.fromarray(arr).save(f"images/input/{name}")

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, (name, arr) in zip(axes, [("original", base)] + list(variants.items())):
    ax.imshow(arr); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Dark / underexposed image → Gamma correction & Histogram Equalization

In [ ]:
def load(name):
    return np.array(Image.open(f"images/input/{name}").convert("RGB"))

def show(images, titles, figsize_per=4.2):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per*n, figsize_per))
    if n == 1: axes = [axes]
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(t, fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()

original = load("original.png")


In [ ]:
dark = load("dark.png")
dark_gamma = gamma_correction(dark, gamma=0.45)
dark_he = histogram_equalization(dark)

show([dark, dark_gamma, dark_he],
     ["Dark (input)", "Gamma correction (γ=0.45)", "Histogram equalization"])

print(f"RMS contrast  before: {rms_contrast(dark):.2f}   after (gamma): {rms_contrast(dark_gamma):.2f}   after (HE): {rms_contrast(dark_he):.2f}")
print(f"Entropy       before: {entropy(dark):.2f}   after (gamma): {entropy(dark_gamma):.2f}   after (HE): {entropy(dark_he):.2f}")


**Observation:** Gamma correction (γ<1) expands the dark tonal range
smoothly, preserving natural-looking gradients. Global Histogram
Equalization boosts contrast more aggressively but can introduce visible
**banding** in smooth gradients (e.g. the sky) because it forces the
histogram to be (approximately) uniform.

## 4. Bright / overexposed image → Gamma correction & Contrast stretching

In [ ]:
bright = load("bright.png")
bright_gamma = gamma_correction(bright, gamma=2.2)
bright_stretch = contrast_stretching(bright, 1, 99)

show([bright, bright_gamma, bright_stretch],
     ["Bright (input)", "Gamma correction (γ=2.2)", "Contrast stretching"])

print(f"RMS contrast  before: {rms_contrast(bright):.2f}   after (gamma): {rms_contrast(bright_gamma):.2f}   after (stretch): {rms_contrast(bright_stretch):.2f}")


**Observation:** With γ>1, gamma correction compresses the bright end
of the range, recovering detail lost to clipping/over-exposure. Contrast
stretching re-maps the (compressed) percentile range back across the full
0–255 span.

## 5. Low-contrast image → Contrast stretching, Histogram Equalization, CLAHE

In [ ]:
lowc = load("low_contrast.png")
lowc_stretch = contrast_stretching(lowc, 2, 98)
lowc_he = histogram_equalization(lowc)
lowc_clahe = clahe(lowc, clip_limit=3.0)

show([lowc, lowc_stretch, lowc_he, lowc_clahe],
     ["Low-contrast (input)", "Contrast stretching", "Histogram equalization", "CLAHE"])

for name, out in [("stretch", lowc_stretch), ("HE", lowc_he), ("CLAHE", lowc_clahe)]:
    print(f"{name:8s}  RMS contrast: {rms_contrast(out):6.2f}   entropy: {entropy(out):.2f}")
print(f"{'input':8s}  RMS contrast: {rms_contrast(lowc):6.2f}   entropy: {entropy(lowc):.2f}")


**Observation:** All three methods substantially increase contrast.
Global histogram equalization achieves the biggest *numerical* contrast gain
but at the cost of banding artifacts; **CLAHE** trades a smaller contrast
gain for locally-adaptive, more natural-looking results, since it limits
contrast amplification per tile.

## 6. Blurred image → Spatial sharpening & Frequency-domain high-pass boosting

This section directly implements the *"Sharpening revisited"* and
*"Sharpen filter"* concepts from the lecture: `sharpened = original +
alpha*(original - blurred)`, its Laplacian-kernel equivalent, and the
frequency-domain high-boost filter `G(u,v) = F(u,v)·H(u,v)`.

In [ ]:
blurred = load("blurred.png")
blur_unsharp  = unsharp_mask(blurred, sigma=2.5, amount=2.0)
blur_laplace  = laplacian_sharpen(blurred, weight=1.2)
blur_freq_hp  = freq_highpass(blurred, d0=25, kind="gaussian", boost=1.5)

show([blurred, blur_unsharp, blur_laplace],
     ["Blurred (input)", "Unsharp masking", "Laplacian sharpening"])
show([blurred, np.stack([blur_freq_hp]*3, -1)],
     ["Blurred (input)", "Frequency-domain high-boost (Gaussian HPF, D0=25)"])

print(f"PSNR vs. clean original — blurred: {psnr(original, blurred):.2f} dB")
print(f"PSNR vs. clean original — unsharp: {psnr(original, blur_unsharp):.2f} dB")
print(f"PSNR vs. clean original — laplacian: {psnr(original, blur_laplace):.2f} dB")
print(f"PSNR vs. clean original — freq high-boost: {psnr(original, np.stack([blur_freq_hp]*3,-1)):.2f} dB")


**Observation:** Sharpening cannot perfectly undo a Gaussian blur
(that would require true deconvolution), but it *perceptually* restores
edge crispness by boosting high-frequency content. Overly aggressive
high-pass boosting can amplify noise/ringing (visible as haloing around
edges) and can even *reduce* PSNR relative to the blurred image, which is
why the sharpening amount/D0 must be tuned per image.

## 7. Noisy image (salt & pepper) → Median filter vs. Mean / Gaussian filters

This reproduces the *"Median filters"* / *"Local Statistic Filters"* slide
comparison directly.

In [ ]:
noisy = load("noisy.png")
noisy_median = median_filter(noisy, ksize=3)
noisy_mean   = mean_filter(noisy, ksize=3)
noisy_gauss  = gaussian_filter(noisy, sigma=1.2)

show([noisy, noisy_median, noisy_mean, noisy_gauss],
     ["Noisy (salt & pepper)", "Median filter (3x3)", "Mean filter (3x3)", "Gaussian filter"])

for name, out in [("median", noisy_median), ("mean", noisy_mean), ("gaussian", noisy_gauss)]:
    print(f"{name:8s}  PSNR vs clean original: {psnr(original, out):.2f} dB")
print(f"{'input':8s}  PSNR vs clean original: {psnr(original, noisy):.2f} dB")


**Observation:** The **median filter** removes salt-and-pepper
impulsive noise almost completely while preserving edges, because it
replaces each pixel with the *median* (not the mean) of its neighborhood —
outlier impulse values are simply discarded rather than averaged in. Mean
and Gaussian filters, being linear, only *smear* the noise across
neighboring pixels rather than removing it, which is visible as residual
speckling plus unwanted blurring of real edges.

## 7b. Frequency-domain view: Fourier spectra & Low/High-pass filtering

Matches the *"FT Pair Example"* and *"Low-pass & High-pass Filtering"*
slides: blur concentrates energy near the DC/low-frequency center of the
spectrum, while noise spreads energy into high frequencies.

In [ ]:
spec_orig  = freq_spectrum_image(original)
spec_blur  = freq_spectrum_image(blurred)
spec_noisy = freq_spectrum_image(noisy)
show([spec_orig, spec_blur, spec_noisy],
     ["Spectrum: original", "Spectrum: blurred\n(energy concentrated at center)",
      "Spectrum: noisy\n(energy spread to high-f)"])

lpf_demo = freq_lowpass(noisy, d0=35, kind="gaussian")
hpf_demo = freq_highpass(blurred, d0=25, kind="butterworth", order=2, boost=1.3)
show([noisy, lpf_demo], ["Noisy (input)", "Gaussian Low-Pass Filter (D0=35)\n(denoises but blurs)"])
show([blurred, hpf_demo], ["Blurred (input)", "Butterworth High-Pass boost (D0=25, n=2)"])


## 8. Bonus: fully automatic enhancement (`auto_enhance`)

`diagnose()` inspects mean brightness, RMS contrast, an impulsive-noise
estimate, and Laplacian variance (a standard blur metric) to classify an
arbitrary image, and `auto_enhance()` then routes it to the matching
correction — useful as a one-call "auto-fix" utility.

**Try it on your own photo:** run the cell below in Colab, click *Choose
Files*, and upload any image (dark, backlit, hazy, blurry phone photo,
etc.).

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    user_path = list(uploaded.keys())[0]
    user_img = np.array(Image.open(user_path).convert("RGB"))
    enhanced, problem = auto_enhance(user_img)
    show([user_img, enhanced], [f"Your image (diagnosed: {problem})", "Auto-enhanced"])
except ImportError:
    print("google.colab not available (not running in Colab) — skipping upload demo.")
    print("You can still call auto_enhance(your_numpy_image) directly.")


In [ ]:
# Demo across all 5 synthetic degraded images
for name in ["dark.png", "bright.png", "low_contrast.png", "blurred.png", "noisy.png"]:
    im = load(name)
    out, problem = auto_enhance(im)
    show([im, out], [f"{name} (diagnosed: {problem})", "auto_enhance() output"])


## 9. Save all outputs (for the GitHub report package)

In [ ]:
import shutil

def save(arr, name):
    Image.fromarray(np.asarray(arr).astype(np.uint8)).save(f"images/output/{name}")

save(dark_gamma, "dark_enhanced_gamma.png")
save(dark_he, "dark_enhanced_histeq.png")
save(bright_gamma, "bright_enhanced_gamma.png")
save(bright_stretch, "bright_enhanced_stretch.png")
save(lowc_stretch, "low_contrast_enhanced_stretch.png")
save(lowc_he, "low_contrast_enhanced_histeq.png")
save(lowc_clahe, "low_contrast_enhanced_clahe.png")
save(blur_unsharp, "blurred_enhanced_unsharp.png")
save(blur_laplace, "blurred_enhanced_laplacian.png")
save(blur_freq_hp, "blurred_enhanced_freqHPF.png")
save(noisy_median, "noisy_enhanced_median.png")
save(noisy_mean, "noisy_enhanced_mean.png")
save(noisy_gauss, "noisy_enhanced_gaussian.png")

shutil.make_archive("image_enhancement_outputs", "zip", "images")
print("Saved all outputs and zipped them into image_enhancement_outputs.zip")

try:
    from google.colab import files
    files.download("image_enhancement_outputs.zip")
except ImportError:
    print("(Not in Colab — zip is available in the local file browser instead.)")


---
## Summary

| Degradation | Best technique in this notebook | Why |
|---|---|---|
| Dark | Gamma correction (γ<1) | Smooth tonal expansion, no banding |
| Bright | Gamma correction (γ>1) / contrast stretch | Recovers clipped highlight detail |
| Low contrast | CLAHE | Locally adaptive, avoids over-amplification |
| Blurred | Unsharp masking | Best perceptual sharpness gain without heavy ringing |
| Noisy (salt & pepper) | Median filter | Removes impulses instead of averaging them in |

See `report/REPORT.md` in the repository for the full written analysis,
including quantitative metrics (PSNR / RMS contrast / entropy) for every
method tested.
